In [ ]:
import os
import requests
import tiktoken
import numpy as np
import random
import torch
import math

In [89]:
input_file_path = './data/tinyshakespeare/input.txt'

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

enc = tiktoken.get_encoding('gpt2')
train_ids = torch.tensor(enc.encode_ordinary(train_data), dtype=torch.long)
val_ids = torch.tensor(enc.encode_ordinary(val_data), dtype=torch.long)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens: {len(val_ids):,}")

train tokens: 301,966
val tokens: 36,059


In [90]:
train_ids[2]

tensor(25)

In [91]:
len(enc._mergeable_ranks) + len(enc._special_tokens), enc.n_vocab

(50257, 50257)

In [92]:
T = block_size = 32
vocab_size = enc.n_vocab
L = n_layer = 3
h = n_head = 4
d_m = n_embd = 96
d_k = int(d_m / h)
d_v = int(d_m / h)
d_ff = 384  # 4 * d_m
B = batch_size = 8
# dropout = 0.1

In [108]:
E = torch.randn((vocab_size, d_m), requires_grad=True)
W_Q = torch.randn((d_m, d_k * h), requires_grad=True)  # big ass matrix so multiply it back
W_K = torch.randn((d_m, d_k * h), requires_grad=True)
W_V = torch.randn((d_m, d_v * h), requires_grad=True)
M = torch.triu(torch.ones((T, T)) * -torch.inf, diagonal=1)
W_attn_out = torch.randn((h * d_v, d_m), requires_grad=True)
W_1 = torch.randn((d_m, d_ff), requires_grad=True)
b_1 = torch.randn((1, d_ff), requires_grad=True)
W_2 = torch.randn((d_ff, d_m), requires_grad=True)
b_2 = torch.randn((1, d_m), requires_grad=True)
W_O = torch.randn((d_m, vocab_size), requires_grad=True)
b_O = torch.randn((1, vocab_size), requires_grad=True)

In [102]:
def dprint(*args, **kwargs):
    debug = False
    if debug:
        print(*args, **kwargs)

In [107]:
for e in range(100):
    Xs = []
    Ys = []
    for _ in range(batch_size):
        i = random.randrange(len(train_ids) - T)
        x = train_ids[i : i + T]
        y = train_ids[i + 1 : i + T + 1]
        Xs.append(x)
        Ys.append(y)
    X = torch.stack(Xs)
    dprint(f"X = {X.shape}")
    Y = torch.stack(Ys)  # (B, T)
    dprint(f"Y = {Y.shape}")
    # forward
    X1 = E[X]
    dprint(f"Emb = {X1.shape}")
    # positional encoding
    Q = X1 @ W_Q
    K = X1 @ W_K
    V = X1 @ W_V
    Q = Q.reshape((B, T, h, d_k)).permute(0, 2, 1, 3)
    K = K.reshape((B, T, h, d_k)).permute(0, 2, 1, 3)
    V = V.reshape((B, T, h, d_v)).permute(0, 2, 1, 3)
    dprint(f"Q = {Q.shape}")
    dprint(f"K = {K.shape}")
    dprint(f"V = {V.shape}")
    A = torch.softmax((Q @ K.transpose(-2, -1)) / math.sqrt(d_k) + M, dim=-1) @ V
    A = A.permute(0, 2, 1, 3)  # (B, T, h, d_v)
    A = A.reshape(B, T, h * d_v)  # (B, T, h * d_v)
    A = A @ W_attn_out
    dprint(f"A = {A.shape}")
    X2 = A + X1
    X3 = (X2 - X2.mean(dim=-1, keepdim=True)) / torch.sqrt(X2.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)
    X4 = torch.relu(X3 @ W_1 + b_1)
    X5 = X4 @ W_2 + b_2
    X6 = X5 + X3
    X7 = (X6 - X6.mean(dim=-1, keepdim=True)) / torch.sqrt(X6.var(dim=-1, keepdim=True, unbiased=False) + 1e-5)
    logits = X7 @ W_O + b_O
    probs = torch.softmax(logits, dim=-1)
    loss = -torch.log(probs.gather(dim=-1, index=Y.unsqueeze(-1)).squeeze(-1)).mean()
    print(f"loss: {loss.item()}")

    # backward
    for param in (E, W_Q, W_K, W_V, W_attn_out, W_1, b_1, W_2, b_2, W_O, b_O):
        param.grad = None
    loss.backward()

    # update
    lr = 0.1
    for param in (E, W_Q, W_K, W_V, W_attn_out, W_1, b_1, W_2, b_2, W_O, b_O):
        param.data -= lr * param.grad

loss: 8.006465911865234
loss: 7.8211669921875
loss: 8.033092498779297
loss: 8.102302551269531
loss: 7.8831868171691895
loss: 7.548818588256836
loss: 8.34635066986084
loss: 7.913658142089844
loss: 7.991764068603516
loss: 7.70723819732666
loss: 7.343227863311768
loss: 7.716203212738037
loss: 7.618408203125
loss: 8.12488842010498
loss: 7.5943779945373535
loss: 7.856809139251709
loss: 7.722245216369629
loss: 7.544000625610352
loss: 7.870776653289795
loss: 7.835650444030762
loss: 7.860047340393066
loss: 8.058418273925781
loss: 7.727609634399414
loss: 7.755224227905273
loss: 7.456865310668945
loss: 7.98320198059082
loss: 7.800247669219971
loss: 8.009285926818848
loss: 7.615920543670654
loss: 7.655778884887695
loss: 7.139607906341553
loss: 7.672177791595459
loss: 7.090926647186279
loss: 7.26862907409668
loss: 7.156433582305908
loss: 7.5790886878967285
loss: 7.8050994873046875
loss: 7.14610481262207
loss: 7.2736496925354
loss: 7.707223892211914
loss: 7.275300025939941
loss: 7.405402183532715
l

In [68]:
K.shape, K.T.shape, K.transpose(1, 2).shape

(torch.Size([8, 32, 96]), torch.Size([96, 32, 8]), torch.Size([8, 96, 32]))

In [94]:
tokens = [enc.decode([int(i)]) for i in probs.argmax(dim=1)]

In [95]:
tokens

[' Citizen',
 ':',
 '\n',
 '\n',
 ' we',
 ' proceed',
 ' any',
 ' further',
 ',',
 ' speak',
 ' me',
 ' speak',
 '.',
 '\n',
 '\n',
 '\n',
 ':',
 '\n',
 '\n',
 'ak',
 ',',
 ' speak',
 '.',
 '\n',
 '\n',
 '\n',
 ' Citizen',
 ':',
 '\n',
 '\n',
 ' are',
 ' all']

In [90]:
X

tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389])

In [91]:
[enc.decode([int(i)]) for i in X]

['First',
 ' Citizen',
 ':',
 '\n',
 'Before',
 ' we',
 ' proceed',
 ' any',
 ' further',
 ',',
 ' hear',
 ' me',
 ' speak',
 '.',
 '\n',
 '\n',
 'All',
 ':',
 '\n',
 'Spe',
 'ak',
 ',',
 ' speak',
 '.',
 '\n',
 '\n',
 'First',
 ' Citizen',
 ':',
 '\n',
 'You',
 ' are']

In [97]:
len(train_ids)

301966

In [101]:
t = random.randrange(len(train_ids - T))

In [102]:
t

80849

In [106]:
vocab_size

50257